In [1]:
import os

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.master("spark://spark-master:7077")
    .appName("Testing Azure")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-azure:3.3.1,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3,org.apache.iceberg:iceberg-aws-bundle:1.4.3,org.apache.hadoop:hadoop-aws:3.3.4",
    )
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    )
    .getOrCreate()
)

spark

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-azure added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-09e38194-8faa-4af4-92b2-eaa5f7181f0c;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-azure;3.3.1 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.11 in central
	found com.microsoft.azure#azure-storage;7.0.1 in central
	found com.fasterxml.jackson.core#jackson-core;2.10.5 in central
	found org.slf4j#slf4j-api;1.7.30 in central
	found com.microsoft.azure#azure-keyvault-core;1.0.0 in central
	found com.google.guava#g

In [2]:
# account_name = os.getenv("AZURE_STORAGE_ACCOUNT_NAME")
account_name = "uniquestocksdev"
client_id = os.getenv("AZURE_CLIENT_ID")
tenant_id = os.getenv("AZURE_TENANT_ID")
secret = os.getenv("AZURE_CLIENT_SECRET")
spark.conf.set(
    f"fs.azure.account.auth.type.{account_name}.dfs.core.windows.net", "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{account_name}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{account_name}.dfs.core.windows.net",
    client_id,
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{account_name}.dfs.core.windows.net",
    secret,
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{account_name}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
)

In [2]:
CATALOG_NAME = "uniquestocks_dev"

spark.conf.set(
    f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog"
)
spark.conf.set(
    f"spark.sql.catalog.{CATALOG_NAME}.catalog-impl",
    "org.apache.iceberg.aws.glue.GlueCatalog",
)
spark.conf.set(
    f"spark.sql.catalog.{CATALOG_NAME}.warehouse",
    "s3a://uniquestocks-datalake-dev/curated/iceberg/",
)
spark.conf.set(
    f"spark.sql.catalog.{CATALOG_NAME}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO"
)
spark.conf.set(
    f"spark.sql.catalog.{CATALOG_NAME}.default-namespace", "uniquestocks_dev"
)
spark.conf.set("spark.sql.defaultCatalog", "uniquestocks_dev")

In [18]:
CATALOG_NAME = "uniquestocks"
# DEFAULT_NAMESPACE = "curated"

import os

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")


spark.conf.set(
    f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog"
)
spark.conf.set(
    f"spark.sql.catalog.{CATALOG_NAME}.catalog-impl",
    "org.apache.iceberg.jdbc.JdbcCatalog",
)
# spark.conf.set(f"spark.sql.catalog.{CATALOG_NAME}.default-namespace", DEFAULT_NAMESPACE)
spark.conf.set(
    f"spark.sql.catalog.{CATALOG_NAME}.uri",
    "jdbc:postgresql://uniquestocks-lakehouse-catalog:5432/iceberg",
)
spark.conf.set(f"spark.sql.catalog.{CATALOG_NAME}.jdbc.user", "admin")
spark.conf.set(f"spark.sql.catalog.{CATALOG_NAME}.jdbc.password", "password")
spark.conf.set(
    f"spark.sql.catalog.{CATALOG_NAME}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO"
)
spark.conf.set(
    f"spark.sql.catalog.{CATALOG_NAME}.warehouse",
    "s3a://uniquestocks/data-lake/lakehouse/", 
)

In [ ]:
CATALOG_NAME = "uniquestocks"
# DEFAULT_NAMESPACE = "curated"

import os

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION")

spark.conf.set(
    f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog"
)
spark.conf.set(f"spark.sql.catalog.{CATALOG_NAME}.type","hive")
# spark.conf.set(f"spark.sql.catalog.{CATALOG_NAME}.default-namespace", DEFAULT_NAMESPACE)
spark.conf.set(
    f"spark.sql.catalog.{CATALOG_NAME}.uri",
    "thrift://metastore:9083",
)
# spark.conf.set(
#     f"spark.sql.catalog.{CATALOG_NAME}.s3.endpoint",
#     "http://minio:9000",
# )

spark.conf.set("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY_ID)
spark.conf.set("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
spark.conf.set("spark.hadoop.fs.s3a.endoint.region", AWS_REGION)
spark.conf.set(
    f"spark.sql.catalog.{CATALOG_NAME}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO"
)
spark.conf.set(
    f"spark.sql.catalog.{CATALOG_NAME}.warehouse",
    "s3a://uniquestocks/data-lake/lakehouse", 
)

In [16]:
%%sql
USE uniquestocks

++
||
++
++

In [8]:
%%sql
SELECT last.*, adjusted_close FROM (SELECT
    security_code,
    exchange_code,
    MAX(date) AS date
FROM curated.security_quote WHERE date BETWEEN date_sub('2024-02-16', 14) AND '2024-02-16'
GROUP BY 1, 2 HAVING date = '2024-02-16') AS last LEFT JOIN (SELECT security_code,
    exchange_code, date, adjusted_close FROM curated.security_quote) AS quote USING (security_code,
    exchange_code, date)

24/02/17 17:52:58 WARN DataSourceV2Strategy: Can't translate true to source filter, unsupported expression


security_code,exchange_code,date,adjusted_close
ROVR,NASDAQ,2024-02-16,10.960000038146973
CRWD,NASDAQ,2024-02-16,329.239990234375
GOS,XETRA,2024-02-16,359.29998779296875
CAT,NYSE,2024-02-16,321.9100036621094
IPWR,NASDAQ,2024-02-16,6.800000190734863
CUBI-PE,NYSE,2024-02-16,25.651199340820312
DTC,NYSE,2024-02-16,2.7200000286102295
AHG,NASDAQ,2024-02-16,1.4800000190734863
GRC,NYSE,2024-02-16,37.650001525878906
OLLI,NASDAQ,2024-02-16,77.81999969482422


24/02/17 17:56:25 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-127985ed-f48f-4999-bf36-f1108473569b/userFiles-308aed73-640b-43e3-92d5-e5e420c8930a. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-127985ed-f48f-4999-bf36-f1108473569b/userFiles-308aed73-640b-43e3-92d5-e5e420c8930a
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:173)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:109)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:90)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1126)
	at org.apache.spark.SparkEnv.stop(SparkEnv.scala:108)
	at org.apache.spark.SparkContext.$anonfun$stop$25(SparkContext.scala:2310)
	at org.apac

In [17]:
%%sql
SHOW NAMESPACES;

namespace
curated
default
mapping
transformed


In [5]:
%%sql

INSERT INTO newDB.exchange VALUES ('df', NULL, NULL, NULL, NULL, NULL, NULL)

++
||
++
++

In [21]:
%%sql
CREATE TABLE IF NOT EXISTS curated.security_quote_performance (
        security_code STRING,
        exchange_code STRING,
        current_date DATE,
        reference_date DATE,
        period STRING,
        current_quote FLOAT,
        reference_quote FLOAT,
        performance FLOAT,
        created_at TIMESTAMP,
        updated_at TIMESTAMP
) USING iceberg;

++
||
++
++

In [25]:
%%sql

SHOW TABLES FROM curated;

namespace,tableName,isTemporary
curated,exchange,False
curated,security,False
curated,security_quote,False
curated,fundamental,False
curated,entity,False
curated,entity_isin,False
curated,security_quote_performance,False


In [5]:
%%sql
SELECT * FROM curated.exchange

code,name,operating_mic,currency,country,created_at,updated_at


In [7]:
spark.read.parquet("s3a://uniquestocks/data-lake/temp/ac03cd4fc8144f3380ecc002c5c7cb57.parquet").createOrReplaceTempView("data")

In [8]:
%%sql
INSERT OVERWRITE
        curated.exchange (name, code, operating_mic, currency, country, created_at, updated_at)
    SELECT
        name, code, operating_mic, currency, country, current_timestamp(), NULL
    FROM data;

++
||
++
++

24/02/17 13:09:16 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-6ffe9d4c-c24a-46b2-9d31-6ac17983869e/userFiles-12557516-ded6-4c9c-b9ee-da12afef85cb. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-6ffe9d4c-c24a-46b2-9d31-6ac17983869e/userFiles-12557516-ded6-4c9c-b9ee-da12afef85cb
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:173)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:109)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:90)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:121)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:120)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1126)
	at org.apache.spark.SparkEnv.stop(SparkEnv.scala:108)
	at org.apache.spark.SparkContext.$anonfun$stop$25(SparkContext.scala:2310)
	at org.apac

In [9]:
%%sql
SELECT
    security_quote.exchange_code,
    type,
    MAX(date) AS latest_date,
    MIN(date) AS earliest_date,
    COUNT(DISTINCT security_quote.security_code) AS count_security
FROM
    curated.security_quote
LEFT JOIN curated.security ON security_quote.security_code = security.code AND security_quote.exchange_code = security.exchange_code
GROUP BY
    1, 2
ORDER BY 1, 2;

exchange_code,type,latest_date,earliest_date,count_security
CC,currency,2024-02-11,2010-07-17,2569
LSE,None,2024-02-09,2024-02-02,7
LSE,common_stock,2024-02-09,1968-12-31,3596
LSE,etc,2024-02-09,2024-02-02,19
LSE,etf,2024-02-09,2024-02-02,2724
LSE,fund,2024-02-08,2024-02-02,4
LSE,index,2024-02-08,2024-02-02,1
LSE,preferred_stock,2024-02-09,2024-02-02,2
MU,common_stock,2024-02-09,1998-09-17,2321
NASDAQ,None,2024-02-09,2024-02-02,36


In [17]:
%%sql
--SELECT DISTINCT exchange_code, security_code FROM curated.security_quote

SELECT * FROM curated.security_quote.refs

name,type,snapshot_id,max_reference_age_in_ms,min_snapshots_to_keep,max_snapshot_age_in_ms
main,BRANCH,7946705779407307355,None,None,None


In [13]:
%%sql
SELECT
    MAX(adjusted_close)
FROM
    curated.security_quote
WHERE security_code = 'AAPL'

max(adjusted_close)
198.11000061035156


## AWS


In [2]:
import os

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")

spark.conf.set("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY_ID)
spark.conf.set("spark.hadoop.fs.s3a.awsAccessKeyId", AWS_ACCESS_KEY_ID)
spark.conf.set("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
spark.conf.set("spark.hadoop.fs.s3a.awsSecretAccessKey", AWS_SECRET_ACCESS_KEY)


spark.conf.set(
    "spark.hadoop.fs.s3a.aws.credentials.provider",
    "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
)

In [4]:
spark.read.parquet(
    "s3a://uniquestocks/data-lake/temp/969aea7296c34c98aae4954de1533755"
).createOrReplaceTempView("reference")
spark.read.parquet(
    "s3a://uniquestocks/data-lake/temp/7d879ee36b634d41b283747f7f1dc7d3"
).createOrReplaceTempView("current")

In [26]:
%%sql
SELECT
    current.security_code,
    current.exchange_code,
    current.date AS current_date,
    reference.date as reference_date,
     reference.period,
    current.quote AS current_quote,
    reference.quote AS reference_quote,
    current.quote / reference.quote - 1 AS performance
FROM current
LEFT JOIN reference USING (security_code, exchange_code)

security_code,exchange_code,current_date,reference_date,period,current_quote,reference_quote,performance
BAC-P-Q,NYSE,2024-02-16,2023-02-16,L12M,19.0,18.6200008392334,0.020408117273868376
BAC-P-Q,NYSE,2024-02-16,2023-11-16,L3M,19.0,17.459999084472656,0.08820166072613844
SNDR,NYSE,2024-02-16,2023-02-16,L12M,23.739999771118164,28.492599487304688,-0.16680119756374345
SNDR,NYSE,2024-02-16,2023-11-16,L3M,23.739999771118164,22.743200302124023,0.043828461067594304
AUBAP,NYSE,2024-02-16,2023-02-16,L12M,24.079999923706055,23.33930015563965,0.031736160173055916
AUBAP,NYSE,2024-02-16,2023-11-16,L3M,24.079999923706055,19.648099899291992,0.22556379737125432
LSXMK,NASDAQ,2024-02-16,2023-11-16,L3M,30.010000228881836,26.3700008392334,0.13803561902936434
SOJC,NYSE,2024-02-16,2023-02-16,L12M,24.47610092163086,22.83009910583496,0.0720978830694261
SOJC,NYSE,2024-02-16,2023-11-16,L3M,24.47610092163086,22.946500778198242,0.06665940738493381
TCPC,NASDAQ,2024-02-16,2023-02-16,L12M,11.170000076293945,10.937999725341797,0.021210491568639966


In [57]:


from pyspark.sql.types import StructType, StructField, StringType, DateType
from pyspark.sql.functions import col




# schema = StructType([
#     StructField("LEI", StringType(), True),
#     StructField("Entity.LegalName", StringType(), True),
#     StructField("Entity.LegalForm.EntityLegalFormCode", StringType(), True),
#     StructField("Entity.LegalJurisdiction", StringType(), True),
#     StructField("Entity.LegalAddress.FirstAddressLine", StringType(), True),
#     StructField("Entity.LegalAddress.AddressNumber", StringType(), True),
#     StructField("Entity.LegalAddress.PostalCode", StringType(), True),
#     StructField("Entity.LegalAddress.City", StringType(), True),
#     StructField("Entity.LegalAddress.Country", StringType(), True),
#     StructField("Entity.HeadquartersAddress.FirstAddressLine", StringType(), True),
#     StructField("Entity.HeadquartersAddress.AddressNumber", StringType(), True),
#     StructField("Entity.HeadquartersAddress.City", StringType(), True),
#     StructField("Entity.HeadquartersAddress.PostalCode", StringType(), True),
#     StructField("Entity.HeadquartersAddress.Country", StringType(), True),
#     StructField("Entity.EntityStatus", StringType(), True),
#     StructField("Entity.EntityCreationDate", DateType(), True),
#     StructField("Entity.EntityExpirationDate", DateType(), True),
#     StructField("Entity.EntityExpirationReason", StringType(), True),
#     StructField("Registration.InitialRegistrationDate", DateType(), True),
#     StructField("Registration.RegistrationStatus", StringType(), True),
# ])


# data = spark.read.option("header", True).option("compression", "gzip").schema(schema).csv("s3a://uniquestocks/data-lake/raw/entity/Gleif/2024/01/29/20-15-25_719293/787c3e286f2c45aeb0bbc3fe17b73d06.csv.gz")
data = spark.read.option("header", True).option("compression", "gzip").csv("s3a://uniquestocks/data-lake/raw/entity/Gleif/2024/01/29/20-15-25_719293/787c3e286f2c45aeb0bbc3fe17b73d06.csv.gz")




# )

# data.printSchema()

#            
#            
          

data = data.select(
    col("LEI"),
    col("`Entity.LegalName`"),
    col("`Entity.LegalForm.EntityLegalFormCode`"),
    col("`Entity.LegalJurisdiction`"),
    col("`Entity.LegalAddress.FirstAddressLine`"),
    col("`Entity.LegalAddress.AddressNumber`"),
    col("`Entity.LegalAddress.PostalCode`"),
    col("`Entity.LegalAddress.City`"),
    col("`Entity.LegalAddress.Country`"),

    
        col("`Entity.HeadquartersAddress.FirstAddressLine`"),
        col("`Entity.HeadquartersAddress.AddressNumber`"),
        col("`Entity.HeadquartersAddress.City`"),
        col("`Entity.HeadquartersAddress.PostalCode`"),
        col("`Entity.HeadquartersAddress.Country`"),
        col("`Entity.EntityStatus`"),
        col("`Entity.EntityCreationDate`"),
        col("`Entity.EntityExpirationDate`"),
        col("`Entity.EntityExpirationReason`"),
        col("`Registration.InitialRegistrationDate`"),
        col("`Registration.RegistrationStatus`")
)

data = data.withColumnRenamed("LEI", "lei") \
           .withColumnRenamed("Entity.LegalName", "name") \
            .withColumnRenamed("Entity.LegalForm.EntityLegalFormCode", "legal_form_id") \
            .withColumnRenamed("Entity.LegalJurisdiction", "jurisdiction") \
             .withColumnRenamed("Entity.LegalAddress.FirstAddressLine", "legal_address_street") \
           .withColumnRenamed("Entity.LegalAddress.AddressNumber", "legal_address_street_number") \
           .withColumnRenamed("Entity.LegalAddress.PostalCode", "legal_address_zip_code") \
           .withColumnRenamed("Entity.LegalAddress.City", "legal_address_city") \
           .withColumnRenamed("Entity.LegalAddress.Country", "legal_address_country") \
           .withColumnRenamed("Entity.HeadquartersAddress.FirstAddressLine", "headquarter_address_street") \
           .withColumnRenamed("Entity.HeadquartersAddress.AddressNumber", "headquarter_address_street_number") \
           .withColumnRenamed("Entity.HeadquartersAddress.City", "headquarter_address_city") \
           .withColumnRenamed("Entity.HeadquartersAddress.PostalCode", "headquarter_address_zip_code") \
           .withColumnRenamed("Entity.HeadquartersAddress.Country", "headquarter_address_country") \
           .withColumnRenamed("Entity.EntityStatus", "status") \
           .withColumnRenamed("Entity.EntityCreationDate", "creation_date") \
           .withColumnRenamed("Entity.EntityExpirationDate", "expiration_date") \
           .withColumnRenamed("Entity.EntityExpirationReason", "expiration_reason") \
           .withColumnRenamed("Registration.InitialRegistrationDate", "registration_date") \
           .withColumnRenamed("Registration.RegistrationStatus", "registration_status")


data = data.withColumn("creation_date", (col("creation_date").cast("timestamp"))) \
            .withColumn("expiration_date", (col("expiration_date").cast("timestamp"))) \
            .withColumn("registration_date", (col("registration_date").cast("timestamp")))


data.createOrReplaceTempView("transformed")
data.show()
# data.select("status", "creation_date", "expiration_date", "registration_date").show()



+--------------------+--------------------+-------------+------------+--------------------+---------------------------+----------------------+------------------+---------------------+--------------------------+---------------------------------+------------------------+----------------------------+---------------------------+--------+-------------------+---------------+-----------------+--------------------+-------------------+
|                 lei|                name|legal_form_id|jurisdiction|legal_address_street|legal_address_street_number|legal_address_zip_code|legal_address_city|legal_address_country|headquarter_address_street|headquarter_address_street_number|headquarter_address_city|headquarter_address_zip_code|headquarter_address_country|  status|      creation_date|expiration_date|expiration_reason|   registration_date|registration_status|
+--------------------+--------------------+-------------+------------+--------------------+---------------------------+-------------------

In [63]:
%%sql
SELECT legal_address_country, COUNT(*) FROM curated.entity GROUP BY 1;

legal_address_country,count(1)
LA,14
ID-JB,1
RU-BEL,1
IT-SA,1
LT,2747
UA,201
IT-BO,2
CASCAIS,1
Stargard Szczecinski,1
KRNAGAR,1


In [61]:
%%sql

INSERT OVERWRITE 
    curated.entity
    (
        lei,
        name,
        legal_form_id,
        jurisdiction,
        legal_address_street,
        legal_address_street_number,
        legal_address_zip_code,
        legal_address_city,
        legal_address_country,
        headquarter_address_street,
        headquarter_address_street_number,
        headquarter_address_city,
        headquarter_address_zip_code,
        headquarter_address_country,
        status,
        creation_date,
        expiration_date,
        expiration_reason,
        registration_date,
        registration_status, 
        created_at, 
        updated_at
    )
SELECT
    lei,
    name,
    legal_form_id,
    jurisdiction,
    legal_address_street,
    legal_address_street_number,
    legal_address_zip_code,
    legal_address_city,
    legal_address_country,
    headquarter_address_street,
    headquarter_address_street_number,
    headquarter_address_city,
    headquarter_address_zip_code,
    headquarter_address_country,
    status,
    creation_date,
    expiration_date,
    expiration_reason,
    registration_date,
    registration_status, 
    current_timestamp(),
    NULL
FROM transformed;

++
||
++
++

In [ ]:
%%sql
INSERT OVERWRITE
        fundamental (category, metric, value, currency, period, period_type, published_at, exchange_code, security_code, created_at, updated_at)
    SELECT
        category, metric, value, currency, period, period_type, published_at, exchange_code, security_code, current_timestamp(), NULL
    FROM data;

In [5]:
%%sql
SELECT * FROM fundamental;

24/01/17 18:16:09 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Py4JJavaError: An error occurred while calling o36.sql.
: software.amazon.awssdk.services.glue.model.GlueException: The security token included in the request is invalid. (Service: Glue, Status Code: 400, Request ID: 344951bf-29de-4c80-b86f-27df8357a7cb)
	at software.amazon.awssdk.core.internal.http.CombinedResponseHandler.handleErrorResponse(CombinedResponseHandler.java:125)
	at software.amazon.awssdk.core.internal.http.CombinedResponseHandler.handleResponse(CombinedResponseHandler.java:82)
	at software.amazon.awssdk.core.internal.http.CombinedResponseHandler.handle(CombinedResponseHandler.java:60)
	at software.amazon.awssdk.core.internal.http.CombinedResponseHandler.handle(CombinedResponseHandler.java:41)
	at software.amazon.awssdk.core.internal.http.pipeline.stages.HandleResponseStage.execute(HandleResponseStage.java:40)
	at software.amazon.awssdk.core.internal.http.pipeline.stages.HandleResponseStage.execute(HandleResponseStage.java:30)
	at software.amazon.awssdk.core.internal.http

# Azure


In [9]:
from pyspark.sql import types as t

CONTAINER = "raw"
PATH = "security_quote/EodHistoricalData/2024/02/12/18-48-31_363775"

schema = t.StructType(
    [
        t.StructField("Date", t.DateType(), True),
        t.StructField("exchange", t.StringType(), True),
        t.StructField("security", t.StringType(), True),
        t.StructField("Open", t.DoubleType(), True),
        t.StructField("High", t.DoubleType(), True),
        t.StructField("Low", t.DoubleType(), True),
        t.StructField("Close", t.DoubleType(), True),
        t.StructField("Adjusted_close", t.DoubleType(), True),
        t.StructField("Volume", t.LongType(), True),
    ]
)

data = (
    spark.read
    .schema(schema)
    .option("header", True)
    .csv(f"abfs://{CONTAINER}@{account_name}.dfs.core.windows.net/{PATH}")
)


# data.createOrReplaceTempView("data")

# spark.sql(
#     """
# SELECT
#     Date as date,
#     Open as open,
#     High AS high,
#     Low as low,
#     Close as close,
#     Adjusted_close as adjusted_close,
#     Volume as volume,
#     security AS security_code,
#     exchange AS exchange_code,
#     current_timestamp() AS created_at,
#     NULL AS updated_at
# FROM
#     data
# WHERE
#     date < current_date();
# """
# ).createOrReplaceTempView("transformed")


# spark.sql("""
# INSERT INTO security (code, name, isin, country, currency, exchange_code, typecreated_at, updated_at) SELECT code, name, isin, country, currency, exchange_code, type FROM transformed
# """)

AttributeError: 'list' object has no attribute 'show'

In [11]:
data.createOrReplaceTempView("data")

In [12]:
spark.sql(
    """
SELECT
    Date as date,
    Open as open,
    High AS high,
    Low as low,
    Close as close,
    Adjusted_close as adjusted_close,
    Volume as volume,
    security AS security_code,
    exchange AS exchange_code,
    current_timestamp() AS created_at,
    NULL AS updated_at
FROM
    data
WHERE
    date < current_date();
"""
).createOrReplaceTempView("transformed")

In [13]:
spark.sql(
        """
    MERGE INTO curated.security_quote AS t 
    USING (SELECT * FROM transformed) AS s  
    ON (t.date = s.date AND t.security_code = s.security_code AND t.exchange_code = s.exchange_code)
    WHEN NOT MATCHED THEN 
            INSERT 
                (date, open, high, low, close, adjusted_close, volume, 
                security_code, exchange_code, created_at, updated_at) 
            VALUES
                (s.date, s.open, s.high, s.low, s.close, s.adjusted_close, s.volume, 
                s.security_code, s.exchange_code, s.created_at, NULL)
    """
    )

DataFrame[]

In [3]:
CONTAINER = "temp"
PATH = "7954625faa43418b98ad2d3e9a5195f2"

from pyspark.sql import types as t

schema = t.StructType(
    [
        t.StructField("Date", t.DateType(), True),
        t.StructField("exchange", t.StringType(), True),
        t.StructField("security", t.StringType(), True),
        t.StructField("Open", t.DoubleType(), True),
        t.StructField("High", t.DoubleType(), True),
        t.StructField("Low", t.DoubleType(), True),
        t.StructField("Close", t.DoubleType(), True),
        t.StructField("Adjusted_close", t.DoubleType(), True),
        t.StructField("Volume", t.LongType(), True),
    ]
)

data = (
    spark.read
    .schema(schema)
    .option("header", True)
    # .option("compression", "gzip")
    .csv(f"abfs://{CONTAINER}@{account_name}.dfs.core.windows.net/{PATH}")
).show()

+----------+--------+--------+--------+--------+--------+--------+--------------+------+
|      Date|exchange|security|    Open|    High|     Low|   Close|Adjusted_close|Volume|
+----------+--------+--------+--------+--------+--------+--------+--------------+------+
|2024-02-06|     LSE|    0A00|   70.82|    71.4|    70.0| 71.1211|       71.1211|199583|
|2024-02-06|     LSE|    0A02|   25.64|    25.8|   25.14| 25.3625|       25.3625|130158|
|2024-02-06|     LSE|    0A05|128.4962| 128.999|128.4962|128.8728|      128.8728|    85|
|2024-02-06|     LSE|    0A0C|   27.58|    28.1|    27.5| 27.9371|       27.9371|121140|
|2024-02-06|     LSE|    0A0D|   65.78|   66.78|   65.64|   66.14|         66.14| 96429|
|2024-02-06|     LSE|    0A0F|    4.71|  4.7545|    4.71|  4.7545|        4.7545| 10379|
|2024-02-06|     LSE|    0A0H|   138.0|   140.8|   136.8|138.5073|      138.5073| 36804|
|2024-02-06|     LSE|    0A0I|   89.35|    90.8|    88.9| 89.3286|       89.3286| 14534|
|2024-02-06|     LSE|

In [23]:
CONTAINER = "raw"
PATH = "entity_isin/Gleif/2024/01/22/17-34-39_713708/35760888abc644eebf9bfffffdc89090.zip"


test = spark.sparkContext.addFile(f"abfs://{CONTAINER}@{account_name}.dfs.core.windows.net/{PATH}")
test


24/01/23 18:32:05 WARN SparkContext: The path abfs://raw@uniquestocksdev.dfs.core.windows.net/entity_isin/Gleif/2024/01/22/17-34-39_713708/35760888abc644eebf9bfffffdc89090.zip has been added already. Overwriting of added paths is not supported in the current version.


In [42]:
# spark.sparkContext.addFile("file:///tmp/test2/lei-isin-20230712T070126.csv")

SparkFiles.get("lei-isin-20230712T070126.csv")

'/tmp/spark-09e0b2bb-f2bf-4eb6-b45f-0a2d24f69023/userFiles-f17c0628-fd26-4e16-b4ca-6272dc346151/lei-isin-20230712T070126.csv'

In [67]:
import gzip
content = "Lots of content here"
with gzip.open('/tmp/new/new.csv.gz', 'wb') as f:
    with open("/tmp/test/lei-isin-20230712T070126.csv") as r:
        f.write(read.read())

FileNotFoundError: [Errno 2] No such file or directory: '/tmp/new/new.csv.gz'

In [64]:
from pyspark.sql import types as t
schema = t.StructType(
    [
        t.StructField("LEI", t.StringType(), True),
        t.StructField("ISIN", t.StringType(), True),
    ]
)

data = spark.read.option("header", True).schema(schema).csv("/tmp/test/")
data.show()

+--------------------+------------+
|                 LEI|        ISIN|
+--------------------+------------+
|001GPB6A9XPE8XJICC14|US3158052262|
|00EHHQ2ZHDCFXJCPCL46|US92204Q1031|
|00KLB2PFTM3060S2N216|US4138382027|
|00KLB2PFTM3060S2N216|US4138385749|
|01ERPZV3DOLNXY2MLB90|US531554AA10|
|01ERPZV3DOLNXY2MLB90|US531554AB92|
|01ERPZV3DOLNXY2MLB90|US531554AC75|
|01ERPZV3DOLNXY2MLB90|US531554AD58|
|01ERPZV3DOLNXY2MLB90|US531554AE32|
|01ERPZV3DOLNXY2MLB90|US531554AF07|
|01ERPZV3DOLNXY2MLB90|US531554AG89|
|01ERPZV3DOLNXY2MLB90|US531554AH62|
|01ERPZV3DOLNXY2MLB90|US531554AJ29|
|01ERPZV3DOLNXY2MLB90|US531554AK91|
|01ERPZV3DOLNXY2MLB90|US531554AL74|
|01ERPZV3DOLNXY2MLB90|US531554AM57|
|01ERPZV3DOLNXY2MLB90|US531554AN31|
|01ERPZV3DOLNXY2MLB90|US531554AP88|
|01ERPZV3DOLNXY2MLB90|US531554AQ61|
|01ERPZV3DOLNXY2MLB90|US531554AR45|
+--------------------+------------+
only showing top 20 rows



In [30]:
from pyspark import SparkFiles

file = SparkFiles.get("35760888abc644eebf9bfffffdc89090.zip")
file
# Read the zip file
# rdd = spark.sparkContext.binaryFiles(file)
# rdd

'/tmp/spark-09e0b2bb-f2bf-4eb6-b45f-0a2d24f69023/userFiles-f17c0628-fd26-4e16-b4ca-6272dc346151/35760888abc644eebf9bfffffdc89090.zip'

In [32]:
import zipfile

def unzip_file(zip_filepath, dest_dir):
    with zipfile.ZipFile(zip_filepath, 'r') as zip_ref:
        zip_ref.extractall(dest_dir)

# Usage
unzip_file(file, '/tmp/test2')

In [33]:
file = SparkFiles.get("test2")
file

'/tmp/spark-09e0b2bb-f2bf-4eb6-b45f-0a2d24f69023/userFiles-f17c0628-fd26-4e16-b4ca-6272dc346151/test2'

In [ ]:
%%sql
MERGE INTO security_quote AS t 
USING (SELECT * FROM transformed) AS s  
ON (t.date = s.date AND t.security_code = s.security_code AND t.exchange_code = s.exchange_code)
WHEN NOT MATCHED THEN 
        INSERT (date, open, high, low, close, adjusted_close, volume, security_code, exchange_code, created_at, updated_at) 
        VALUES (s.date, s.open, s.high, s.low, s.close, s.adjusted_close, s.volume, s.security_code, s.exchange_code, s.created_at, NULL)

In [ ]:

# spark.read.option("compression", "gzip").option("header", True).csv(url)

from pyspark import SparkFiles

spark.sparkContext.addFile("https://raw.githubusercontent.com/lukes/ISO-3166-Countries-with-Regional-Codes/master/all/all.csv")
spark.read.csv("file://" + )

In [ ]:
SparkFiles.get("all.csv")

In [22]:
from pyspark.sql import SparkSession
import io
import csv
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType
import zipfile

spark = SparkSession.builder.getOrCreate()

# Read the zip file
rdd = spark.sparkContext.binaryFiles(file)

# Unzip the file and extract the data
def extract_zip(index, iterator):
    for i in iterator:
        name, content = i
        with zipfile.ZipFile(io.BytesIO(content)) as z:
            for filename in z.namelist():
                with z.open(filename) as f:
                    reader = csv.reader(io.TextIOWrapper(f))
                    for row in reader:
                        yield row

# Define your schema
schema = StructType([
    StructField("column1", StringType(), True),
    StructField("column2", StringType(), True),
    # Add more columns here
])

# Apply the function to the RDD
rdd = rdd.mapPartitionsWithIndex(extract_zip)

# Convert the RDD to DataFrame
df = spark.createDataFrame(rdd, schema)

df.show()

24/01/23 18:31:27 WARN TaskSetManager: Lost task 0.0 in stage 13.0 (TID 22) (172.19.0.3 executor 0): java.io.FileNotFoundException: File file:/tmp/spark-09e0b2bb-f2bf-4eb6-b45f-0a2d24f69023/userFiles-f17c0628-fd26-4e16-b4ca-6272dc346151/35760888abc644eebf9bfffffdc89090.zip does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:779)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1100)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:769)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.hadoop.fs.ChecksumFileSystem$ChecksumFSInputChecker.<init>(ChecksumFileSystem.java:160)
	at org.apache.hadoop.fs.ChecksumFileSystem.open(ChecksumFileSystem.java:372)
	at org.apache.hadoop.fs.FileSystem.open(FileSystem.java:976)
	at org.apache.spark.input.PortableDataStream.open(PortableDataStream.scala:195)
	at org.apache.spark

Py4JJavaError: An error occurred while calling o138.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 13.0 failed 4 times, most recent failure: Lost task 0.3 in stage 13.0 (TID 25) (172.19.0.3 executor 0): java.io.FileNotFoundException: File file:/tmp/spark-09e0b2bb-f2bf-4eb6-b45f-0a2d24f69023/userFiles-f17c0628-fd26-4e16-b4ca-6272dc346151/35760888abc644eebf9bfffffdc89090.zip does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:779)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1100)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:769)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.hadoop.fs.ChecksumFileSystem$ChecksumFSInputChecker.<init>(ChecksumFileSystem.java:160)
	at org.apache.hadoop.fs.ChecksumFileSystem.open(ChecksumFileSystem.java:372)
	at org.apa

In [ ]:
%%sql
SELECT exchange_code, COUNT(DISTINCT security_code) FROM security_quote GROUP BY exchange_code;

In [ ]:
from pyspark.sql import types as t


schema = t.StructType(
    [
        t.StructField("Date", t.DateType(), True),
        t.StructField("exchange", t.StringType(), True),
        t.StructField("security", t.StringType(), True),
        t.StructField("Open", t.DoubleType(), True),
        t.StructField("High", t.DoubleType(), True),
        t.StructField("Low", t.DoubleType(), True),
        t.StructField("Close", t.DoubleType(), True),
        t.StructField("Adjusted_close", t.DoubleType(), True),
        t.StructField("Volume", t.LongType(), True),
    ]
)


container = "raw"
path = "security_quote/EodHistoricalData/2024/01/15/20-35-35_905622"


data = (
    spark.read.schema(schema)
    .option("header", True)
    .csv(f"abfs://{container}@{account_name}.dfs.core.windows.net/{path}")
)
data.createOrReplaceTempView("data")

spark.sql(
    """
SELECT
    Date as date,
    Open as open,
    High AS high,
    Low as low,
    Close as close,
    Adjusted_close as adjusted_close,
    Volume as volume,
    security AS security_code,
    exchange AS exchange_code,
    current_timestamp() AS created_at,
    NULL AS updated_at
FROM
    data
WHERE
    date < current_date();
"""
).show()

In [ ]:
%%sql
SELECT * FROM security_quote;

In [ ]:
from pyspark.sql import types as t

container = "raw"
path = "exchange/EodHistoricalData/year=2024/month=01/day=04/20240104-000000__exchange__EodHistoricalData.json"

schema = t.StructType(
    [
        t.StructField("Name", t.StringType(), True),
        t.StructField("Code", t.StringType(), True),
        t.StructField("Country", t.StringType(), True),
        t.StructField("CountryISO2", t.StringType(), True),
        t.StructField("CountryISO3", t.StringType(), True),
        t.StructField("Currency", t.StringType(), True),
        t.StructField("OperatingMIC", t.StringType(), True),
    ]
)


data = spark.read.schema(schema).json(
    f"abfs://{container}@{account_name}.dfs.core.windows.net/{path}"
)
data.show()

In [ ]:
container = "temp"
path = "test.parquet"


data = spark.read.parquet(
    f"abfs://{container}@{account_name}.dfs.core.windows.net/{path}"
)
data.show()

In [ ]:
data.createOrReplaceTempView("data")

os.environ["AWS_DEFAULT_REGION"] = "eu-central-1"
os.environ["AWS_REGION"] = "eu-central-1"

spark.sql(
    """
    SELECT
        Name AS name,
        Code AS code,
        OperatingMIC AS operating_mic,
        IF(Currency='Unknown', NULL, Currency) AS currency,
        IF(CountryISO2='', NULL, CountryISO2) AS country
    FROM data

"""
).createOrReplaceTempView("transformed")

# .write.format("iceberg").mode("overwrite") .save("uniquestocks_dev.uniquestocks_dev.exchange")

In [ ]:
%%sql
SELECT COUNT(*) FROM exchange

In [ ]:
%%sql
DELETE FROM exchange

In [ ]:
spark.sql(
    """
DELETE FROM exchange;
INSERT INTO exchange SELECT * FROM transformed;
"""
)

In [ ]:
%%sql
INSERT OVERWRITE exchange SELECT * FROM transformed;

In [ ]:
container = "temp"
path = "ba66a3afb1d447fdb009a68304177605.parquet"


data = spark.read.parquet(
    f"abfs://{container}@{account_name}.dfs.core.windows.net/{path}"
)
data.createOrReplaceTempView("security_transformed")
data.show()

In [ ]:
%%sql
INSERT OVERWRITE security SELECT * FROM security_transformed;

In [ ]:
%%sql
--DROP TABLE security_quote PURGE;
SELECT * FROM security_quote;

In [ ]:
# account_name ="uniquestocksdatalake"
container = "temp"
path = "e24faa01c15343ea9aa1209364b9b2c9"


data = spark.read.parquet(
    f"abfs://{container}@{account_name}.dfs.core.windows.net/{path}"
).select(
    "date",
    "open",
    "high",
    "low",
    "close",
    "adjusted_close",
    "volume",
    "exchange_code",
    "security_code",
    "created_at",
)

from pyspark.sql.functions import lit


data = data.withColumn("updated_at", lit(None).cast("timestamp"))

data.createOrReplaceTempView("security_quote_transformed")
data.printSchema()

In [ ]:
%%sql
INSERT OVERWRITE security_quote SELECT * FROM security_quote_transformed;

### Snapshot


In [ ]:
%%sql
SELECT * FROM uniquestocks_dev.uniquestocks_dev.exchange.snapshots;

### History


In [ ]:
%%sql
SELECT * FROM uniquestocks_dev.uniquestocks_dev.security_quote.history;

In [ ]:
spark.stop()

# PyIceberg


In [ ]:
import os

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")

from pyiceberg.catalog import load_catalog

catalog = load_catalog(
    "uniquestocks_dev",
    **{
        "s3.access-key-id": AWS_ACCESS_KEY_ID,
        "aws_access_key_id": AWS_ACCESS_KEY_ID,
        "s3.secret-access-key": AWS_SECRET_ACCESS_KEY,
        "aws_secret_access_key": AWS_SECRET_ACCESS_KEY,
        "region_name": "eu-central-1",
        "type": "glue",
    }
)
catalog.load_table("uniquestocks_dev.security_quote").scan().to_pandas()

In [ ]:
# Slowly Changing Dimension (SCD Type 2)
MERGE INTO s3lakehouse.blog.customer_base as b
USING
( SELECT null as custkey_match, custkey, name, state, zip, cust_since, last_update_dt,'Y' as active_ind,current_timestamp as end_dt
FROM s3lakehouse.blog.customer_land
UNION ALL
SELECT
custkey as custkey_match,custkey, name, state, zip, cust_since, last_update_dt,active_ind,end_dt
FROM s3lakehouse.blog.customer_base
WHERE custkey IN
(SELECT custkey FROM s3lakehouse.blog.customer_land where active_ind = 'Y')
) as scdChangeRows
ON (b.custkey = scdChangeRows.custkey and b.custkey = scdChangeRows.custkey_match)
WHEN MATCHED and b.active_ind = 'Y' THEN
UPDATE SET end_dt = current_timestamp,active_ind = 'N'
WHEN NOT MATCHED THEN
        INSERT (custkey, name, state, zip, cust_since,last_update_dt,active_ind,end_dt)
            VALUES(scdChangeRows.custkey, scdChangeRows.name, scdChangeRows.state, scdChangeRows.zip,
                    scdChangeRows.cust_since,scdChangeRows.last_update_dt,'Y',null);